#**GitHub Link**

https://github.com/Satyam-G-Kulkarni/Integrating-MLOps-for-Predictive-and-Recommender-Systems-in-Travel



#**Importing Necessary Libraries**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb

from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install flask_ngrok pyngrok sentence-transformers

In [ ]:
from pyngrok import ngrok
from flask_ngrok import run_with_ngrok
from flask import Flask, render_template, request, jsonify
!ngrok authtoken "5PRC5XE2BKY55FHRWFUHKZBQKHPVI7XC"



#**Data Ingestion**

In [ ]:
flight_df=pd.read_csv("/content/drive/MyDrive/Projects/Productionization of ML Systems/flights.csv",on_bad_lines='skip')
hotel_df=pd.read_csv("/content/drive/MyDrive/Projects/Productionization of ML Systems/hotels.csv",on_bad_lines='skip')
user_df=pd.read_csv("/content/drive/MyDrive/Projects/Productionization of ML Systems/users.csv",on_bad_lines='skip')

In [ ]:
flight_df.head()

create a function price/km and apply it in dataframe and create new column

In [ ]:
def price_km(x,y):
  price_per_km=x/y

  return price_per_km



In [ ]:
flight_df.apply


#**Data Mining and Exploration**

In [ ]:
# Total no. of rows in the dataset

df_list=[flight_df,hotel_df,user_df]
df_name_list=['flight_df','hotel_df','user_df']

for i in df_name_list:
  if i == 'flight_df':
    print(f'Total no. of rows in {i}: {len(df_list[0])}')
  elif i == 'hotel_df':
    print(f'Total no. of rows in {i}: {len(df_list[1])}')
  else:
    print(f'Total no. of rows in user_df: {len(df_list[2])}')



In [ ]:
# Overall Information of Dataset

for i in df_name_list:
  if i == 'flight_df':
    print(f'Information about {i}')
    print(df_list[0].info())
  elif i == 'hotel_df':
    print(f'Information about {i}')
    print(df_list[1].info())
  else:
    print(f'Information about {i}')
    print(df_list[2].info())

In [ ]:
flight_df.describe(include='object')

In [ ]:
# Summary statistics of numeric columns

for i in df_name_list:
  if i == 'flight_df':
    print(i)
    print(df_list[0].describe())
  elif i == 'hotel_df':
    print(i)
    print(df_list[1].describe())
  else:
    print(i)
    print(df_list[2].describe())

In [ ]:
# Check for Missing Values

for i in df_name_list:
  if i == 'flight_df':
    print(f'Total no. of null rows in {i}: {df_list[0].isnull().sum()}')
  elif i == 'hotel_df':
    print(f'Total no. of null rows in {i}: {df_list[1].isnull().sum()}')
  else:
    print(f'Total no. of null rows in user_df: {df_list[2].isnull().sum()}')

In [ ]:
# Check for Duplicate rows

for i in df_name_list:
  if i == 'flight_df':
    print(f'Total no. of duplicate rows in {i}: {df_list[0].duplicated().sum()}')
  elif i == 'hotel_df':
    print(f'Total no. of duplicate rows in {i}: {df_list[1].duplicated().sum()}')
  else:
    print(f'Total no. of duplicate rows in user_df: {df_list[2].duplicated().sum()}')

##**Correcting the data type of date coulmn to datetime**

In [ ]:
#flight_df1= flight_df.merge(user_df,how='outer',left_on='userCode',right_on='code')

In [ ]:
# Converting date coulmn data type into datetime

flight_df['date'] = pd.to_datetime(flight_df['date'])

# Extracting WeekNo., Month, Year, Weekday from date column

flight_df['week_day'] = flight_df['date'].dt.weekday
flight_df['month'] = flight_df['date'].dt.month
flight_df['week_no'] = flight_df['date'].dt.isocalendar().week
flight_df['year'] = flight_df['date'].dt.year
flight_df['day'] = flight_df['date'].dt.day

In [ ]:
flight_df.head()

In [ ]:
flight_filtered_df= flight_df[['from','to','flightType','agency','time','distance','day','month','year','week_day','week_no','price']]
#flight_filtered_df= flight_df.copy()
flight_filtered_df.head()

In [ ]:
flight_filtered_df.shape



#**Exploratory Data Analysis**

## **Univariate Analysis**

###**Numerical Coulmn Distribution**

Check for skewness in the distribution of numerical Columns

In [ ]:
# Select only the numerical columns
numerical_columns = flight_filtered_df.select_dtypes(include=['number'])

# Calculate the skewness for the numerical columns
skewness = numerical_columns.skew()

print(skewness)


In [ ]:
skewness.plot(kind='bar')


From the above result, we can check which variable is normally distributed and which is not.

The variables with skewness > 1 price are highly positively skewed.

The variables with skewness < -1 are highly negatively skewed.

The variables with 0.5 < skewness < 1 are moderately positively skewed.

The variables with -0.5 < skewness < -1 are moderately negatively skewed.

And, the variables with -0.5 < skewness < 0.5 are symmetric i.e normally distributed such as symboling, carheight, boreration, peakrpm, highwaympg.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(flight_filtered_df['time'], bins=30, kde=True)
plt.title('Histogram of Travel Time')
plt.xlabel('Travel Time (hrs)')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.show()

Inference from the plot:

The histogram shows the distribution of travel time, with the x-axis representing the travel time and the y-axis representing the frequency (i.e., the number of occurrences) of these travel time.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(flight_filtered_df['distance'], bins=30, kde=True)
plt.title('Histogram of Travel Distance')
plt.xlabel('Travel Distance (kms)')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.show()

Inference from the plot:

The histogram shows the distribution of travel distance, with the x-axis representing the travel distance and the y-axis representing the frequency (i.e., the number of occurrences) of these travel distance.

In [ ]:
# Flight Price Distribution

plt.figure(figsize=(10, 6))
sns.histplot(flight_filtered_df['price'], kde=True, color='lightblue')
plt.title('Flight Price Distribution')
plt.xlabel('Flight Price')
plt.ylabel('Frequency')
plt.show()

Inference from the plot:

The majority of the prices seem to be concentrated in a relatively narrow range, indicating a common price range for these travel packages.

There is a long tail on the right side of the distribution, suggesting that there are a few travel packages with significantly higher prices compared to the majority.

The distribution appears to be right-skewed, with prices tapering off as they increase.

Overall, this plot provides insight into the distribution of prices, which is valuable information for understanding the pricing structure of the travel packages in the dataset.

In [ ]:
# Plot histograms for numeric columns
numeric_cols = flight_filtered_df.select_dtypes(include=['int64', 'float64']).columns
flight_filtered_df[numeric_cols].hist(bins=20, figsize=(12, 8))
plt.show()

### **Check for Outliers in Numerical Coulmns using Box-plot**

If the distribution of numerical column follows normal distribution, then use std to handle outliers.

Otherwise we will use IQR technique

In [ ]:
# Plot box plots for numeric columns to detect outliers
plt.figure(figsize=(12, 8))
sns.boxplot(data=flight_filtered_df['price'])
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(data=flight_filtered_df['distance'])
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(data=flight_filtered_df['time'])
plt.xticks(rotation=45)
plt.show()

### **Distribution of Categories in Categorical Coulmns using a Count-plot**

In [ ]:
flight_filtered_df.info()

In [ ]:
sns.countplot(x='flightType',data=flight_filtered_df,palette = "Set2")

In [ ]:
sns.countplot(x='agency',data=flight_filtered_df,palette = "Set2")

In [ ]:
sns.countplot(x='month',data=flight_filtered_df,palette = "Set2")

### **Distribution of Categories in Categorical Coulmns using a pie-chart**

In [ ]:
# Get the counts of each flight type
flight_type_counts = flight_filtered_df['flightType'].value_counts()

# Display the counts of each class in 'Flight Type'
print(flight_type_counts)

# Create the pie chart
plt.figure(figsize=(8, 8))
plt.pie(flight_type_counts, labels=flight_type_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Flight Types')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

In [ ]:
# Get the counts of each agency
agency_type_counts = flight_filtered_df['agency'].value_counts()

# Display the counts of each class in 'Agency Type'
print(agency_type_counts)

# Create the pie chart
plt.figure(figsize=(8, 8))
plt.pie(agency_type_counts, labels=agency_type_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Agency')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

Inference from the plot:

The pie chart illustrates the distribution of selected package types in the dataset. The data is categorized into five package types: Deluxe, Standard, Premium, Luxury, and Budget.

Deluxe and Standard are the most common package types, each making up a significant portion of the dataset.
Premium is also fairly common, but slightly less frequent than Deluxe and Standard.
Luxury packages are less common but still represent a substantial portion of the dataset.
Budget packages are the least common among the selected types, comprising a relatively small percentage of the dataset.

In [ ]:
# Get the counts of each destination_city_counts
destination_city_counts = flight_filtered_df['to'].value_counts()

# Display the counts of each class in 'Destination Cities'
print(agency_type_counts)

# Create the pie chart
plt.figure(figsize=(8, 8))
plt.pie(destination_city_counts, labels=destination_city_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Destination City')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

### **Distribution of Values in Numerical Coulmns using a pie-chart**

In [ ]:
# Get the counts of each flight type
week_day_type_counts = flight_filtered_df['week_day'].value_counts()

# Display the counts of each class in 'Week days'
print(week_day_type_counts)

# Create the pie chart
plt.figure(figsize=(8, 8))
plt.pie(week_day_type_counts, labels=week_day_type_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Week_days')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

In [ ]:
sns.countplot(x='week_day',data=flight_filtered_df,palette = "Set2")

In [ ]:
# Get the counts of each flight type
year_counts = flight_filtered_df['year'].value_counts()

# Display the counts of each class in 'Week days'
print(year_counts)

# Create the pie chart
plt.figure(figsize=(8, 8))
plt.pie(year_counts, labels=year_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Year')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

In [ ]:
sns.countplot(x='year',data=flight_filtered_df,palette = "Set2")

## **Bivariate Analysis**

### **Check for Linear relationship btwn Independent numerical Variable and Target variable**

In [ ]:
# Price vs. Travel Time

plt.figure(figsize=(10, 6))
sns.scatterplot(data=flight_filtered_df, x='time', y='price', color='purple', alpha=0.5)
plt.title('Flight Price vs. Travel Time')
plt.xlabel('Travel Time')
plt.ylabel('Flight Price')
plt.show()

In [ ]:
# Price vs. Travel Distance

plt.figure(figsize=(10, 6))
sns.scatterplot(data=flight_filtered_df, x='distance', y='price', color='purple', alpha=0.5)
plt.title('Flight Price vs. Travel Distance')
plt.xlabel('Travel Distance')
plt.ylabel('Flight Price')
plt.show()


1. **Price vs. Hotel Ratings Plot**:
   - The plot examines the relationship between "Hotel Ratings" and "Per Person Price" for travel packages.
   - Each point represents a travel package, with its price on the y-axis and hotel ratings on the x-axis.

2. **Inference**:
   - There doesn't seem to be a strong linear correlation between hotel ratings and per person price.
   - Most data points are scattered across the plot, indicating that hotel ratings alone may not be a dominant factor in determining the price of travel packages.
   - However, there are some clusters of points, suggesting that specific rating ranges may have an impact on price within certain regions.

### **Check for Outliers in Categorical Coulmns**

In [ ]:
# Box Plots:
# Visualize the distribution of prices per agency type:

plt.figure(figsize=(12, 6))
sns.boxplot(data=flight_filtered_df, x='agency', y='price')
plt.xticks(rotation=90)
plt.xlabel('Agency Type')
plt.ylabel('Flight Price')
plt.title('Price Distribution by Agency Type')
plt.show()

In [ ]:
# Box Plots:
# Visualize the distribution of prices per Flight type:

plt.figure(figsize=(12, 6))
sns.boxplot(data=flight_filtered_df, x='flightType', y='price')
plt.xticks(rotation=90)
plt.xlabel('Flight Type')
plt.ylabel('Flight Price')
plt.title('Price Distribution by Flight Type')
plt.show()

In [ ]:
# Box Plots:
# Visualize the distribution of prices per Destination Cities:

plt.figure(figsize=(12, 6))
sns.boxplot(data=flight_filtered_df, x='to', y='price')
plt.xticks(rotation=90)
plt.xlabel('Destination Cities')
plt.ylabel('Flight Price')
plt.title('Price Distribution by Destination Cities')
plt.show()

**Inference of the plot**:

1. **Variability**: The box plots reveal varying levels of price variability within different package types. Some package types have a wider range of prices (larger boxes), while others have narrower ranges (smaller boxes).

2. **Outliers**: Outliers, represented as individual points beyond the "whiskers" of the boxes, can be seen in several package types. These outliers indicate some extreme price values within those categories.

3. **Median Prices**: The horizontal line inside each box represents the median price for each package type. It's a quick way to compare the typical prices across different categories.

4. **Package Type Impact**: The plot provides a visual sense of how package type influences price distribution. Some package types consistently have higher or lower prices than others.

###**Visualize the distribution of prices per Categorical Coulmns using bar-plot**

In [ ]:
sns.barplot(x='flightType',y='price', data=flight_filtered_df)

The mean most expensive flight Class Type was the FirstClass, followed by Premium Class and Economic Class.

In [ ]:
sns.barplot(x='agency',y='price', data=flight_filtered_df)

The mean most expensive flight tickets booked by agency was Flyingdrops, followed by Rainbow Class and Cloudfy agency.

## **Multi-Variate Analysis**

In [ ]:
# Bar Plots:
# Explore the relationship between the agency and flight types:

plt.figure(figsize=(12, 6))
sns.barplot(data=flight_filtered_df, x='agency', y='price', hue='flightType')
plt.xticks(rotation=90)
plt.xlabel('Agency')
plt.ylabel('Flight Price')
plt.title('Flight Price by Agency and Flight Type')
plt.show()

In [ ]:
# Stacked Bar Plot:
# Show the distribution of flight types by agency using a stacked bar plot:

flight_package_counts = flight_filtered_df.groupby(['agency', 'flightType']).size().unstack(fill_value=0)
flight_package_counts.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.xlabel('Agency')
plt.ylabel('Count')
plt.title('Distribution of Flight Types by Agency (Stacked Bar Plot)')
plt.xticks(rotation=45)
plt.show()

Inference of the plot:

The plot shows that the agency Flyingdrops is quite popular in booking only first class tickets to its customers and no bookings in any other flight type"

On the contrary, rest two agencies are helping their customers to book their tickets in various flight Class apart from First Class. Both these agencies work in pretty much similar price ranges.

This information suggests that Flyingdrops only deals with HNI or Premium Customers and there is no diverse option apart from FirstClass Type for the customers.

Inference from the plot:

The stacked bar plot visualizes the distribution of package types by start city in the dataset.

In each start city, the stacked bars represent the different package types (Deluxe, Standard, Premium, Luxury, Budget).
For example, in New Delhi Deluxe and Standard packages are more prevalent
This plot allows you to see how the distribution of package types varies across both start cities, providing insights into regional preferences for package types.

In [ ]:
# Word Clouds:
# If you want to visualize common words in text columns like "Boarding City," you can create a word cloud:

from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(flight_df['from']))
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Boarding City')
plt.show()

In [ ]:
# Word Clouds:
# If you want to visualize common words in text columns like "Destination," you can create a word cloud:

from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(flight_df['to']))
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Destination')
plt.show()

Inference from the plot:

The word cloud plot for the "Destination" column visually represents the most common words found in the text data. From the plot, it appears that some of the frequently mentioned destinations include Shimla, Manali, Thekkady, Alleppey, Munnar, and New Delhi. These destinations seem to be popular or commonly mentioned in the dataset.


# **Feature Engineering**

In [ ]:
# renaming the Column name
flight_filtered_df.rename(columns={"to":"destination"},inplace=True)

In [ ]:
# Creating a new feature using distance and time columns
flight_filtered_df['flight_speed']=round(flight_filtered_df['distance']/flight_filtered_df['time'],2)

In [ ]:
flight_filtered_df.head()

In [ ]:
sns.boxplot(x=flight_filtered_df['flight_speed'])

In [ ]:
Q1= np.percentile(flight_filtered_df['flight_speed'],25)
Q3= np.percentile(flight_filtered_df['flight_speed'],75)
IQR= Q3-Q1


In [ ]:
lower_bound= Q1- IQR*1.5
upper_bound= Q3+IQR*1.5

The distribution of flight speed coulmn in the above graph is indicating a left skew due to outliers ,these outliers we can see in box plot occuring below the flight speed of 383km/hr. It may be not considered as outlier in this case,because these are valid data poits and these speed occured due to bad weather condition most probably.





In [ ]:
# Select only the numerical columns
numerical_columns = flight_filtered_df.select_dtypes(include=['number'])

# Calculate the skewness for the numerical columns
skewness = numerical_columns.skew()

print(skewness)

In [ ]:
df=flight_filtered_df.copy()
df.describe()

In [ ]:
flight_filtered_df.info()

**One-hot encoding**

In [ ]:
# Example of one-hot encoding
flight_filtered_df = pd.get_dummies(flight_filtered_df, columns=['from','destination','flightType','agency'])

In [ ]:
flight_filtered_df.shape



# **Feature Selection**

##**Feature Selection Using Statistical Test ANNOVA F-Test**

In [ ]:
final_df1= flight_filtered_df.drop(columns=['time','flight_speed','month','year','distance'],axis=1)

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif

# Assuming df is your DataFrame with features and target variable 'target'

# Separate features (X) and target variable (y)
X = final_df1.drop('price', axis=1)  # Features
y = final_df1['price']                # Target variable

# Select top k features based on chi-squared test for categorical features
#categorical_features = X.select_dtypes(include=['object', 'category']).columns
num_features = X.select_dtypes(include=['number']).columns

# ANOVA F-test for numerical features
f_selector = SelectKBest(score_func=f_classif, k='all')
f_selector.fit(X[num_features], y)

# Get indices of top k features for both categorical and numerical features

numerical_top_indices = f_selector.get_support(indices=True)

# Combine top indices
top_indices = list(numerical_top_indices)

# Get top k feature names
top_features = X.columns[top_indices]

# Display top k features
print("Top k selected features:")
print(top_features)


In [ ]:
#Multicollinearity
# Define the calc_vif function
def calc_vif(X):
    vif = pd.DataFrame()
    vif["variables"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif

# Filter to include only numeric columns
numeric_columns = flight_filtered_df.select_dtypes(include=[np.number])

# Filter columns that are in top_features
filtered_columns = [col for col in numeric_columns.columns if col in top_features]

# Create a DataFrame with the filtered numeric columns
vif_df = flight_filtered_df[filtered_columns]

# Convert all values to numeric and cast to float64
vif_df = vif_df.apply(pd.to_numeric, errors='coerce').astype(np.float64)

# Handle missing values by filling with the mean
vif_df = vif_df.fillna(vif_df.mean())

# Debugging: Check the data types and values in vif_df
print("Data types in vif_df:")
print(vif_df.dtypes)
print("First few rows of vif_df:")
print(vif_df.head())

# Ensure the DataFrame contains only numeric data and no NaN values
if not np.isfinite(vif_df.values).all():
    print("Non-numeric or NaN values detected.")
    for column in vif_df.columns:
        if not np.isfinite(vif_df[column].values).all():
            print(f"Problematic column: {column}")
            print(vif_df[column].unique())

# Proceed only if there are no non-numeric or NaN values
assert np.isfinite(vif_df.values).all(), "There are non-numeric or NaN values in the DataFrame."

# Calculate VIF
vif_result = calc_vif(vif_df)

print(vif_result)


In [ ]:
X.rename(columns={'from_Sao Paulo (SP)':'from_Sao_Paulo (SP)','from_Rio de Janeiro (RJ)':'from_Rio_de_Janeiro (RJ)','from_Campo Grande (MS)':'from_Campo_Grande (MS)',
                                  'destination_Sao Paulo (SP)':'destination_Sao_Paulo (SP)','destination_Rio de Janeiro (RJ)':'destination_Rio_de_Janeiro (RJ)','destination_Campo Grande (MS)':'destination_Campo_Grande (MS)'},inplace=True)

In [ ]:
features_ordering=['from_Florianopolis (SC)',
 'from_Sao_Paulo (SP)',
 'from_Salvador (BH)',
 'from_Brasilia (DF)',
 'from_Rio_de_Janeiro (RJ)',
 'from_Campo_Grande (MS)',
 'from_Aracaju (SE)',
 'from_Natal (RN)',
 'from_Recife (PE)',
 'destination_Florianopolis (SC)',
 'destination_Sao_Paulo (SP)',
 'destination_Salvador (BH)',
 'destination_Brasilia (DF)',
 'destination_Rio_de_Janeiro (RJ)',
 'destination_Campo_Grande (MS)',
 'destination_Aracaju (SE)',
 'destination_Natal (RN)',
 'destination_Recife (PE)',
 'flightType_economic',
 'flightType_firstClass',
 'flightType_premium',
 'agency_Rainbow',
 'agency_CloudFy',
 'agency_FlyingDrops',
 'week_no',
 'week_day',
 'day']

In [ ]:
#Ordering features based on flask output
final_features_1= X[features_ordering]

In [ ]:
X1= final_features_1

# Target variable
y1 = y

# Split the data into training and testing sets
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size=0.2, random_state=42)

scaler_new = StandardScaler()

X_train1 = scaler_new.fit_transform(X_train1)
X_test1 = scaler_new.transform(X_test1)


In [ ]:
final_features_1.columns

#**Model Building, Training, Cross-Validation, Hyperparameter Tuning & Evaluation**

##**Linear Regression**

###**Baseline Model**

In [ ]:
# Create and train the Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train1, y_train1)

# Make predictions on the test set
y_pred1 = linear_model.predict(X_test1)

# Evaluate the model
mae_lr = mean_absolute_error(y_test1, y_pred1)
mse_lr = mean_squared_error(y_test1, y_pred1)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test1, y_pred1)

# Calculate adjusted R-squared for linear regression
n = X_test1.shape[0]
p = X_test1.shape[1]


# Calculate adjusted R-squared
adj_r2_lr = 1 - ((1 - r2_lr) * (n - 1) / (n - p - 1))


plt.scatter(y_test1, y_test1, c='b', label='Actual', alpha=0.5)  # Blue for actual
plt.scatter(y_test1, y_pred1, c='r', label='Predicted', alpha=0.5)  # Red for predicted
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs. Predicted Values")
plt.legend(loc='upper left')
plt.grid(True)
plt.show()



###**Model Evaluation**

In [ ]:
model_result = pd.DataFrame([['Linear Regression Baseline', mse_lr,rmse_lr,mae_lr, r2_lr,adj_r2_lr]],
               columns = ['Model', 'MSE', 'RMSE', 'MAE', 'R2','adj_r2'])

model_result

###**Hyperparameter Tuning Linear**

In [ ]:
# Define hyperparameters and their potential values
param_grid_linear = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

# Create a GridSearchCV object for Linear Regression
grid_lr = GridSearchCV(linear_model, param_grid_linear, cv=3, scoring='neg_mean_squared_error')
grid_lr.fit(X_train1, y_train1)

# Get the best hyperparameters for Linear Regression
best_linear_reg = grid_lr.best_estimator_

# Perform grid search
best_linear_reg.fit(X_train1, y_train1)



In [ ]:
# Print the best hyperparameters
print("Best Hyperparameters for Linear Regression:", grid_lr.best_estimator_)

### **Model Evaluation**

In [ ]:
# Evaluate the model
y_pred_linear_tuned = best_linear_reg.predict(X_test1)

mae_linear_tuned = mean_absolute_error(y_test1, y_pred_linear_tuned )
mse_linear_tuned = mean_squared_error(y_test1, y_pred_linear_tuned )
rmse_linear_tuned = np.sqrt(mse_linear_tuned)
r2_linear_tuned = r2_score(y_test1, y_pred_linear_tuned )
adj_r2_linear_tuned = 1 - ((1 - r2_linear_tuned) * (n - 1) / (n - p - 1))

model = pd.DataFrame([['Linear Regression Tuned',  mse_linear_tuned,rmse_linear_tuned,mae_linear_tuned, r2_linear_tuned,adj_r2_linear_tuned]],
               columns = ['Model', 'MSE', 'RMSE', 'MAE', 'R2','adj_r2'])

model_result = pd.concat([model_result,model],axis=0,ignore_index = True)
model_result

##**Decision Tree**

###**Baseline Model**

In [ ]:
# Initialize the Decision Tree Regressor model
decision_tree_reg = DecisionTreeRegressor(random_state=42)

# Fit the model on the training data
decision_tree_reg.fit(X_train1, y_train1)

# Make predictions on the test data
y_pred_decision = decision_tree_reg.predict(X_test1)

# Evaluate the model
mse_dt = mean_squared_error(y_test1, y_pred_decision)
rmse_dt = mse_dt**0.5
mae_dt = mean_absolute_error(y_test1, y_pred_decision)
r2_dt = r2_score(y_test1, y_pred_decision)
adj_r2_dt = 1 - ((1 - r2_dt) * (n - 1) / (n - p - 1))

plt.scatter(y_test1, y_test1, c='b', label='Actual', alpha=0.5)  # Blue for actual
plt.scatter(y_test1, y_pred_decision, c='r', label='Predicted', alpha=0.5)  # Red for predicted
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs. Predicted Values")
plt.legend(loc='upper left')
plt.grid(True)
plt.show()


### **Model Evaluation**

In [ ]:
model = pd.DataFrame([['Decision Tree Baseline',  mse_dt,rmse_dt,mae_dt, r2_dt,adj_r2_dt]],
               columns = ['Model', 'MSE', 'RMSE', 'MAE', 'R2','adj_r2'])

model_result = pd.concat([model_result,model],axis=0,ignore_index = True)
model_result

###**Hyperparameter tunning**

In [ ]:
# Define hyperparameters and their potential values
param_grid_dt = {
    'criterion': ['friedman_mse', 'squared_error'],
    'max_depth': [15,30,45],
    'min_samples_split': [2,3,5],
    'max_features': [15,27,'sqrt'],
    'ccp_alpha' :[1,2]
}


# Create a GridSearchCV object for Decision Tree Regressor
grid_search_dt = GridSearchCV(decision_tree_reg, param_grid_dt, cv=3, scoring='neg_mean_squared_error')
grid_search_dt.fit(X_train1, y_train1)

# Get the best hyperparameters for Decision Tree Regressor
best_decision_tree_model = grid_search_dt.best_estimator_

In [ ]:
grid_search_dt.best_estimator_

In [ ]:
# Print the best hyperparameters
print("Best Hyperparameters for Decision Tree:", grid_search_dt.best_estimator_)

### **Model Evaluation**

In [ ]:
# Evaluate the best model
y_pred_dt_tuned = best_decision_tree_model.predict(X_test1)

mae_dt_tuned = mean_absolute_error(y_test1, y_pred_dt_tuned)
mse_dt_tuned = mean_squared_error(y_test1, y_pred_dt_tuned)
rmse_dt_tuned = np.sqrt(mse_dt_tuned)
r2_dt_tuned = r2_score(y_test1, y_pred_dt_tuned)
adj_r2_dt_tuned = 1 - ((1 - r2_dt_tuned) * (n - 1) / (n - p - 1))

model = pd.DataFrame([['Decision Tree Tuned',  mse_dt_tuned,rmse_dt_tuned,mae_dt_tuned, r2_dt_tuned,adj_r2_dt_tuned]],
               columns = ['Model', 'MSE', 'RMSE', 'MAE', 'R2','adj_r2'])

model_result = pd.concat([model_result,model],axis=0,ignore_index = True)
model_result

###**Feature Importance Decision Tree**

In [ ]:
# Get feature importances
feature_importances_dt_reg = best_decision_tree_model.feature_importances_

In [ ]:
import matplotlib.pyplot as plt

# Define feature names
feature_names = X1.columns.tolist()  # Assuming X_train is a DataFrame

# Zip feature names and importances
feature_importances_sorted = sorted(zip(feature_names, feature_importances_dt_reg), key=lambda x: x[1], reverse=True)
sorted_features, sorted_importances = zip(*feature_importances_sorted)

# Create bar plot
plt.figure(figsize=(10, 6))
plt.bar(range(len(sorted_importances)), sorted_importances, align='center')
plt.xticks(range(len(sorted_importances)), sorted_features, rotation=90)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Decision Tree Regressor Feature Importance (Sorted)')
plt.show()


##**Random Forest**

###**Baseline Model**

In [ ]:
# Create and train the Linear Regression model
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train1, y_train1)

# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test1)

# Evaluate the model
mae_rf = mean_absolute_error(y_test1,y_pred_rf)
mse_rf = mean_squared_error(y_test1, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test1, y_pred_rf)
adj_r2_rf = 1 - ((1 - r2_rf) * (n - 1) / (n - p - 1))



plt.scatter(y_test1, y_test1, c='b', label='Actual', alpha=0.5)  # Blue for actual
plt.scatter(y_test1, y_pred_rf, c='r', label='Predicted', alpha=0.5)  # Red for predicted
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs. Predicted Values")
plt.legend(loc='upper left')
plt.grid(True)
plt.show()

### **Model Evaluation**

In [ ]:
model = pd.DataFrame([['Random Forest Baseline',  mse_rf,rmse_rf,mae_rf, r2_rf,adj_r2_rf]],
               columns = ['Model', 'MSE', 'RMSE', 'MAE', 'R2','adj_r2'])

model_result = pd.concat([model_result,model],axis=0,ignore_index = True)
model_result

###**Hyperparameter tunning**

In [ ]:
# Define hyperparameters and their potential values
param_grid_rf = {
    'n_estimators': [50],
    'max_depth': [10],
    'min_samples_split': [2, 5,10],
    'max_features': ['sqrt',27],
    'n_jobs': [2]


}

# Create a GridSearchCV object for Random Forest Regression
grid_search_rf = GridSearchCV(rf_model, param_grid_rf, cv=3, scoring='neg_mean_squared_error')
grid_search_rf.fit(X_train1, y_train1)

# Get the best hyperparameters for Random Forest Regression
best_rf_model = grid_search_rf.best_estimator_


In [ ]:
grid_search_rf.best_estimator_

In [ ]:
# Print the best hyperparameters
print("Best Hyperparameters for Random Forest:", grid_search_rf.best_estimator_)




### **Model Evaluation**

In [ ]:
# Evaluate the best model
y_pred_rf_tuned = best_rf_model.predict(X_test1)

mae_rf_tuned = mean_absolute_error(y_test1, y_pred_rf_tuned)
mse_rf_tuned = mean_squared_error(y_test1, y_pred_rf_tuned)
rmse_rf_tuned = np.sqrt(mse_rf_tuned)
r2_rf_tuned = r2_score(y_test1, y_pred_rf_tuned)
adj_r2_rf_tuned = 1 - ((1 - r2_rf_tuned) * (n - 1) / (n - p - 1))

model = pd.DataFrame([['Random Forest Tuned',  mse_rf_tuned,rmse_rf_tuned,mae_rf_tuned, r2_rf_tuned,adj_r2_rf_tuned]],
               columns = ['Model', 'MSE', 'RMSE', 'MAE', 'R2','adj_r2'])

model_result = pd.concat([model_result,model],axis=0,ignore_index = True)
model_result

###**Feature Importance Random Forest**

In [ ]:
feature_importances_rf_reg = best_rf_model.feature_importances_

In [ ]:
import matplotlib.pyplot as plt

# Define feature names
feature_names = X1.columns.tolist()  # Assuming X_train is a DataFrame

# Zip feature names and importances
feature_importances_sorted = sorted(zip(feature_names, feature_importances_rf_reg), key=lambda x: x[1], reverse=True)
sorted_features, sorted_importances = zip(*feature_importances_sorted)

# Create bar plot
plt.figure(figsize=(10, 6))
plt.bar(range(len(sorted_importances)), sorted_importances, align='center')
plt.xticks(range(len(sorted_importances)), sorted_features, rotation=90)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance (Sorted)')
plt.show()


#**Benchmark Models**

In [ ]:
#Benchmark Model on the basis on RMSE metric
model_result[model_result['RMSE']==model_result['RMSE'].min()]

In [ ]:
#Benchmark Model on the basis on Adjusted R2 square metric
model_result[model_result['adj_r2']==model_result['adj_r2'].max()]

In [ ]:
#Benchmark Models on the basis of RMSE and adjusted R2 square metrics were as follows

filtered_result = model_result[(model_result['adj_r2'] == model_result['adj_r2'].max()) | (model_result['RMSE'] == model_result['RMSE'].min())]
filtered_result

In [ ]:
plt.figure(figsize=(10, 6))

# Linear Regression
plt.scatter(y_test1, y_pred_linear_tuned, label='Linear Regression', alpha=0.6)

# Decision Tree Regressor
plt.scatter(y_test1, y_pred_dt_tuned, label='Decision Tree', alpha=0.6)

# Random Forest Regressor
plt.scatter(y_test1, y_pred_rf_tuned, label='Random Forest', alpha=0.6)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs. Predicted Values")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

# Linear Regression Residuals
residuals_lr = y_test1 - y_pred_linear_tuned
plt.scatter(y_pred_linear_tuned, residuals_lr, label='Linear Regression', alpha=0.6)

# Random Forest Regressor Residuals
residuals_rf = y_test1 - y_pred_rf_tuned
plt.scatter(y_pred_rf_tuned, residuals_rf, label='Random Forest', alpha=0.6)

# Decision Tree Regressor Residuals
residuals_dt = y_test1 -y_pred_dt_tuned
plt.scatter(y_pred_dt_tuned, residuals_dt, label='Decision Tree', alpha=0.6)

plt.axhline(0, color='black', linestyle='--', lw=2)
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# Linear Regression Residuals Distribution
plt.hist(residuals_lr, bins=30, alpha=0.6, label='Linear Regression')

# Random Forest Regressor Residuals Distribution
plt.hist(residuals_rf, bins=30, alpha=0.6, label='Random Forest')

# Decision Tree Regressor Residuals Distribution
plt.hist(residuals_dt, bins=30, alpha=0.6, label='Decision Tree')

plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Distribution of Residuals")
plt.legend()
plt.show()

##**Feature Selection Using Multicollinearity and Vif Score**

In [ ]:
#Multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor
def calc_vif(X):

    # Calculating VIF
    vif = pd.DataFrame()
    vif["variables"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    return(vif)


In [ ]:
final_df= flight_filtered_df.drop(columns=['flight_speed','time','month','year','distance'],axis=1)

In [ ]:
correlation_matrix = final_df.corr()
k = 27 # Replace with the desired number of top features
selected_features = correlation_matrix['price'].abs().sort_values(ascending=False).index[1:k+1]

In [ ]:
# Create a subset of the DataFrame with the selected features
subset_df = final_df[selected_features]

# Calculate the correlation matrix for the selected features
correlation_matrix = subset_df.corr()

# Set up the plot figure size
plt.figure(figsize=(12, 8))

# Create a heatmap of the correlation matrix
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")

# Set the plot title
plt.title('Correlation Matrix of Selected Features')

# Show the plot
plt.show()

In [ ]:
# Filter to include only numeric columns
numeric_columns = flight_filtered_df.select_dtypes(include=[np.number])

# Filter columns that are in top_features
filtered_columns = [col for col in numeric_columns.columns if col in top_features]

# Create a DataFrame with the filtered numeric columns
vif_df = flight_filtered_df[filtered_columns]

# Convert all values to numeric and cast to float64
vif_df = vif_df.apply(pd.to_numeric, errors='coerce').astype(np.float64)

# Handle missing values by filling with the mean
vif_df = vif_df.fillna(vif_df.mean())

# Debugging: Check the data types and values in vif_df
print("Data types in vif_df:")
print(vif_df.dtypes)
print("First few rows of vif_df:")
print(vif_df.head())

# Ensure the DataFrame contains only numeric data and no NaN values
if not np.isfinite(vif_df.values).all():
    print("Non-numeric or NaN values detected.")
    for column in vif_df.columns:
        if not np.isfinite(vif_df[column].values).all():
            print(f"Problematic column: {column}")
            print(vif_df[column].unique())

# Proceed only if there are no non-numeric or NaN values
assert np.isfinite(vif_df.values).all(), "There are non-numeric or NaN values in the DataFrame."

# Calculate VIF
vif_result = calc_vif(vif_df)

print(vif_result)

# **Flask Building**

Things we need to do:

**Flask Web Application**: We create a Flask web application to serve as an API endpoint.

**Load Trained Model**: We load a trained regression model (you should replace this with your actual model loading code).

**Prediction Function**: We define a function (predict_value) to make predictions using the loaded model.

**User Input Processing**: In the /predict route, we collect user input data from an HTML form, including price, travel details, and other parameters.

**Data Preparation**: We prepare the user input data and convert it into a format suitable for making predictions.

**Prediction**: Using the loaded model, we make predictions based on the user's input.

**JSON Response**: We return the prediction result as a JSON response to the API request.

**HTML Form**: If the request is a GET request, we display an HTML form to collect user input.

**Styling**: You can customize the HTML form and add CSS styling to make it visually appealing.

**Run the Flask App**: The Flask app is run, allowing users to access it and receive predictions through API requests.

In [ ]:
import pickle
pickle.dump(scaler_new,open('/content/drive/MyDrive/Projects/Productionization of ML Systems/scaling.pkl','wb'))
pickle.dump(best_rf_model,open('/content/drive/MyDrive/Projects/Productionization of ML Systems/rf_model.pkl','wb'))

In [ ]:
pickle.dump(best_linear_reg,open('/content/drive/MyDrive/Projects/Productionization of ML Systems/lr_model.pkl','wb'))
pickle.dump(best_decision_tree_model,open('/content/drive/MyDrive/Projects/Productionization of ML Systems/dt_model.pkl','wb'))

In [ ]:
with open("/content/drive/MyDrive/Projects/Productionization of ML Systems/scaling.pkl", "wb") as f:
    pickle.dump(scaler_new, f)

In [ ]:
scaler_model=pickle.load(open('/content/drive/MyDrive/Projects/Productionization of ML Systems/scaling.pkl','rb'))
rf_model=pickle.load(open('/content/drive/MyDrive/Projects/Productionization of ML Systems/rf_model.pkl','rb'))

In [ ]:
# Create a function for prediction
def predict_price(input_data, model,scaler):
    # Prepare the input data

    # Initialize an empty DataFrame
    df_input2 = pd.DataFrame([input_data])

    # Independent features
    X = df_input2

    # Scale the data using the same scaler used during training
    X = scaler.transform(X)

    # Make predictions using the trained Decision model
    y_pred = model.predict(X)

    return y_pred[0]

##**Testing Pickled model**

In [ ]:
df_input = pd.read_csv('/content/drive/MyDrive/Projects/Productionization of ML Systems/travel_capstone/flights.csv',on_bad_lines='skip')


df_input['date'] = pd.to_datetime(df_input['date'])
df_input['day'] = df_input['date'].dt.day
df_input['week_no'] = df_input['date'].dt.isocalendar().week
df_input['week_day'] = df_input['date'].dt.weekday

# renaming the Column name
df_input.rename(columns={"to":"destination"},inplace=True)

# one-hot encoding
df_input = pd.get_dummies(df_input, columns=['from','destination','flightType','agency'])

# renaming the Column name
df_input.rename(columns={'from_Sao Paulo (SP)':'from_Sao_Paulo (SP)','from_Rio de Janeiro (RJ)':'from_Rio_de_Janeiro (RJ)','from_Campo Grande (MS)':'from_Campo_Grande (MS)',
                                  'destination_Sao Paulo (SP)':'destination_Sao_Paulo (SP)','destination_Rio de Janeiro (RJ)':'destination_Rio_de_Janeiro (RJ)','destination_Campo Grande (MS)':'destination_Campo_Grande (MS)'},inplace=True)




Y=df_input.loc[:,['from_Florianopolis (SC)',
 'from_Sao_Paulo (SP)',
 'from_Salvador (BH)',
 'from_Brasilia (DF)',
 'from_Rio_de_Janeiro (RJ)',
 'from_Campo_Grande (MS)',
 'from_Aracaju (SE)',
 'from_Natal (RN)',
 'from_Recife (PE)',
 'destination_Florianopolis (SC)',
 'destination_Sao_Paulo (SP)',
 'destination_Salvador (BH)',
 'destination_Brasilia (DF)',
 'destination_Rio_de_Janeiro (RJ)',
 'destination_Campo_Grande (MS)',
 'destination_Aracaju (SE)',
 'destination_Natal (RN)',
 'destination_Recife (PE)',
 'flightType_economic',
 'flightType_firstClass',
 'flightType_premium',
 'agency_Rainbow',
 'agency_CloudFy',
 'agency_FlyingDrops',
 'week_no',
 'week_day',
 'day']]

In [ ]:
#ordering features based on requirement
Z=df_input[X1.columns]

In [ ]:
#Edited Code
row_index = 10  # Replace with the desired row index

# Access the row from df_input using iloc
input_row = Z.iloc[row_index]

# Create an input dictionary from the selected row
input_data = Z.iloc[row_index].to_dict()
#scaler_model_new1=pickle.load(open('/content/drive/MyDrive/travel_capstone/scaler_new','rb'))
#rf_model_new1=pickle.load(open('/content/drive/MyDrive/travel_capstone/dt_model_new','rb'))
predicted_price = str(round(predict_price(input_data, rf_model,scaler_model),2))
print(f'Predicted Flight Price Per Person: ${predicted_price}')

#**Flask Code**

In [ ]:
!ngrok authtoken "2gorxOm9YlGFf8MoRgQfVzx6RqY_de56fyN9P51du4UayK37"

In [ ]:
#Final Flask Code
# Create a function for prediction




import pandas as pd
import pickle
from flask import Flask, request, jsonify

from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pyngrok import ngrok




def predict_price(input_data, model, scaler):
    # Prepare the input data

    # Initialize an empty DataFrame
    df_input2 = pd.DataFrame([input_data])

    # Independent features
    X = df_input2

    # Scale the data using the same scaler used during training
    X = scaler.transform(X)

    # Make predictions using the trained Decision model
    y_prediction = model.predict(X)

    return y_prediction[0]


app = Flask(__name__)


@app.route('/',
           methods=['GET', 'POST'])
def predict():
    return """

<!DOCTYPE html>
<html>
  <head>
    <title>Flight Price Prediction</title>
    <style>
      body {
        font-family: 'Poppins', sans-serif;
        background-color: #f9f9f9;
        margin: 0;
        padding: 0;
      }

      .container {
        max-width: 800px;
        margin: 0 auto;
        padding: 40px;
        background-color: #ffffff;
        border-radius: 10px;
        box-shadow: 0 10px 20px rgba(0, 0, 0, 0.1);
        text-align: center;
      }

      h1 {
        color: #007BFF;
        font-size: 36px;
        margin-bottom: 20px;
      }

      form {
        text-align: left;
      }

      input[type="text"],
      input[type="number"] {
        width: 100%;
        padding: 15px;
        margin: 15px 0;
        border: none;
        border-bottom: 2px solid #007BFF;
        font-size: 18px;
        background-color: transparent;
        color: #333;
        transition: border-bottom 0.3s ease;
      }

      input[type="text"]:focus,
      input[type="number"]:focus {
        border-bottom: 2px solid #0056b3;
        outline: none;
      }

      input[type="checkbox"],
      input[type="radio"] {
        margin-right: 10px;
      }

      input[type="submit"] {
        background-color: #007BFF;
        color: #fff;
        padding: 15px 30px;
        border: none;
        border-radius: 5px;
        cursor: pointer;
        font-size: 20px;
        transition: background-color 0.3s ease;
      }

      input[type="submit"]:hover {
        background-color: #0056b3;
      }

      p#prediction {
        margin-top: 20px;
        font-size: 24px;
        color: #007BFF;
      }
    </style>
  </head>
  <body>
    <div class="container">
      <h1>Flight Price Prediction</h1>
      <form action="/predict" method="POST">
        <b>Select a Boarding City</b>
        <br />
        <br />
        <input type="radio" id="Aracaju" name="from" value="Aracaju">
        <label for="Aracaju">Aracaju</label>
        <br>
        <input type="radio" id="Brasilia" name="from" value="Brasilia">
        <label for="Brasilia">Brasilia</label>
        <br>
        <input type="radio" id="Campo" name="from" value="Campo_Grande">
        <label for="Campo_Grande">Campo_Grande</label>
        <br>
        <input type="radio" id="Florianopolis" name="from" value="Florianopolis">
        <label for="Florianopolis">Florianopolis</label>
        <br>
        <input type="radio" id="Natal" name="from" value="Natal">
        <label for="Natal">Natal</label>
        <br>
        <input type="radio" id="Recife" name="from" value="Recife">
        <label for="Recife">Recife</label>
        <br>
        <input type="radio" id="Rio_de_Janeiro" name="from" value="Rio_de_Janeiro">
        <label for="Rio_de_Janeiro">Rio_de_Janeiro</label>
        <br>
        <input type="radio" id="Salvador" name="from" value="Salvador">
        <label for="Salvador">Salvador</label>
        <br>
        <input type="radio" id="Sao_Paulo" name="from" value="Sao_Paulo">
        <label for="Sao_Paulo">Sao_Paulo</label>
        <br>
        <br />
        <br />
        <b>Select a Destination City</b>
        <br />
        <br />
        <input type="radio" id="Aracaju" name="Destination" value="Aracaju">
        <label for="Aracaju">Aracaju</label>
        <br>
        <input type="radio" id="Brasilia" name="Destination" value="Brasilia">
        <label for="Brasilia">Brasilia</label>
        <br>
        <input type="radio" id="Campo_Grande" name="Destination" value="Campo_Grande">
        <label for="Campo">Campo_Grande</label>
        <br>
        <input type="radio" id="Florianopolis" name="Destination" value="Florianopolis">
        <label for="Florianopolis">Florianopolis</label>
        <br>
        <input type="radio" id="Natal" name="Destination" value="Natal">
        <label for="Natal">Natal</label>
        <br>
        <input type="radio" id="Recife" name="Destination" value="Recife">
        <label for="Recife">Recife</label>
        <br>
        <input type="radio" id="Rio_de_Janeiro" name="Destination" value="Rio_de_Janeiro">
        <label for="Rio_de_Janeiro">Rio_de_Janeiro</label>
        <br>
        <input type="radio" id="Salvador" name="Destination" value="Salvador">
        <label for="Salvador">Salvador</label>
        <br>
        <input type="radio" id="Sao_Paulo" name="Destination" value="Sao_Paulo">
        <label for="Sao_Paulo">Sao_Paulo</label>
        <br>
        <br />
        <br />
        <b>Select a Flight Type</b>
        <br />
        <br />
        <input type="radio" id="flightType_premium" name="flightType" value="premium">
        <label for="flightType_premium">premium</label>
        <br>
        <input type="radio" id="flightType_economic" name="flightType" value="economic">
        <label for="flightType_economic">economic</label>
        <br>
        <input type="radio" id="flightType_firstClass" name="flightType" value="firstClass">
        <label for="flightType_firstClass">firstClass</label>
        <br />
        <br />
        <b>Select Agency</b>
        <br />
        <br />
        <input type="radio" id="FlyingDrops" name="agency" value="FlyingDrops">
        <label for="FlyingDrops">FlyingDrops</label>
        <br>
        <input type="radio" id="Rainbow" name="agency" value="Rainbow">
        <label for="Rainbow">Rainbow</label>
        <br>
        <input type="radio" id="CloudFy" name="agency" value="CloudFy">
        <label for="CloudFy">CloudFy</label>
        <br />
        <br />
        <label for="day">day:</label>
        <input type="number" name="day" min="1" max="31" placeholder="Travel day" value="5">
        <label for="week_no">week_no:</label>
        <input type="number" name="week_no" min="1" max="53" placeholder="Travel Week No" value="7">
        <label for="week_day">week_day No :</label>
        <input type="number" name="week_day" min="1" max="7" placeholder="Travel Week Day" value="5">
        <input type="submit" value="Predict">
      </form>
      <p id="prediction"></p>
    </div>
  </body>
</html>







    """


@app.route('/predict', methods=['POST'])
def index():
    if request.method == 'POST':
        # Get input data from the form
        boarding = str(request.form.get('from'))
        destination = str(request.form.get('Destination'))
        selected_flight_class = str(request.form.get('flightType'))
        selected_agency = str(request.form.get('agency'))
        week_no = int(request.form.get('week_no'))
        week_day = int(request.form.get('week_day'))
        day = int(request.form.get('day'))

        boarding = 'from_' + boarding
        boarding_city_list = ['from_Florianopolis (SC)',
                              'from_Sao_Paulo (SP)',
                              'from_Salvador (BH)',
                              'from_Brasilia (DF)',
                              'from_Rio_de_Janeiro (RJ)',
                              'from_Campo_Grande (MS)',
                              'from_Aracaju (SE)',
                              'from_Natal (RN)',
                              'from_Recife (PE)']

        destination = 'destination_' + destination
        destination_city_list = ['destination_Florianopolis (SC)',
                                 'destination_Sao_Paulo (SP)',
                                 'destination_Salvador (BH)',
                                 'destination_Brasilia (DF)',
                                 'destination_Rio_de_Janeiro (RJ)',
                                 'destination_Campo_Grande (MS)',
                                 'destination_Aracaju (SE)',
                                 'destination_Natal (RN)',
                                 'destination_Recife (PE)']

        selected_flight_class = 'flightType_' + selected_flight_class
        class_list = ['flightType_economic',
                      'flightType_firstClass',
                      'flightType_premium']

        selected_agency = 'agency_' + selected_agency
        agency_list = ['agency_Rainbow',
                       'agency_CloudFy',
                       'agency_FlyingDrops']

        travel_dict = dict()

        for city in boarding_city_list:
            if city[:-5] != boarding:
                travel_dict[city] = 0
            else:
                travel_dict[city] = 1
        for city in destination_city_list:
            if city[:-5] != destination:
                travel_dict[city] = 0
            else:
                travel_dict[city] = 1
        for flight_class in class_list:
            if flight_class != selected_flight_class:
                travel_dict[flight_class] = 0
            else:
                travel_dict[selected_flight_class] = 1
        for agency in agency_list:
            if agency != selected_agency:
                travel_dict[agency] = 0
            else:
                travel_dict[selected_agency] = 1
        travel_dict['week_no'] = week_no
        travel_dict['week_day'] = week_day
        travel_dict['day'] = day

        scaler_model_new=pickle.load(open('/content/drive/MyDrive/Projects/Productionization of ML Systems/scaling.pkl','rb'))
        rf_model_new=pickle.load(open('/content/drive/MyDrive/Projects/Productionization of ML Systems/rf_model.pkl','rb'))
        # Perform prediction using the custom_input dictionary
        predicted_price = str(round(predict_price(travel_dict, rf_model, scaler_model), 2))
        # print(f'Predicted Flight Price Per Person: ${predicted_price}')

        return jsonify({'prediction': predicted_price})


if __name__ == '__main__':
    # Start ngrok tunnel
    public_url = ngrok.connect(addr="5000")
    print(" * Running on", public_url)
    app.run()
